In [7]:
import os

import pandas as pd
import geopandas as gpd
from tqdm import tqdm

In [8]:
# CELL 2 — Load Toronto ADA data

ada_csv = "../data/ada/toronto-ada.csv"
ada_gpkg = "../data/ada/toronto-ada.gpkg"

# Load ADA CSV data
df_ada = pd.read_csv(ada_csv, encoding='latin')

# Load Toronto ADA geometries
gdf_ada = gpd.read_file(ada_gpkg)

print(f"CSV rows: {len(df_ada)}, Geometries: {len(gdf_ada)}")

CSV rows: 55800, Geometries: 279


In [9]:
# CELL 3 — Extract NOCS and NAICS data per ADA

# Define the characteristic IDs
noc_id = 2254
naics_id = 2278

# Columns of interest
count_col = "C1_COUNT_TOTAL"
rate_col = "C10_RATE_TOTAL"

# Pivot CSV data for NOCS
df_nocs = df_ada[df_ada["CHARACTERISTIC_ID"] == noc_id][["GEO_NAME", count_col, rate_col]].copy()
df_nocs = df_nocs.rename(columns={
    count_col: "arts_nocs_count",
    rate_col: "arts_nocs_pct"
})

# Pivot CSV data for NAICS
df_naics = df_ada[df_ada["CHARACTERISTIC_ID"] == naics_id][["GEO_NAME", count_col, rate_col]].copy()
df_naics = df_naics.rename(columns={
    count_col: "arts_naics_count",
    rate_col: "arts_naics_pct"
})

# Merge NOCS and NAICS data
df_merge = pd.merge(df_nocs, df_naics, on="GEO_NAME", how="outer")

print(f"Merged stats for {len(df_merge)} ADAs")

Merged stats for 279 ADAs


In [13]:
df_merge

,GEO_NAME,arts_nocs_count,arts_nocs_pct,arts_naics_count,arts_naics_pct
0,35200001,70.0,2.2,45.0,1.4
1,35200002,115.0,1.7,70.0,1.1
2,35200003,55.0,1.3,25.0,0.6
3,35200004,50.0,1.4,45.0,1.3
4,35200005,175.0,3.5,95.0,1.9
...,...,...,...,...,...
274,35200336,965.0,6.2,275.0,1.8
275,35202000,870.0,8.2,325.0,3.1
276,35202001,350.0,5.6,110.0,1.8
277,35202003,430.0,5.0,140.0,1.6


In [14]:
# CELL 4 — Join statistics with geometries

# Ensure the key columns have the same type
gdf_ada["ADAUID"] = gdf_ada["ADAUID"].astype(str)
df_merge["GEO_NAME"] = df_merge["GEO_NAME"].astype(str)

# Merge with GeoDataFrame
gdf_final = gdf_ada.merge(df_merge, left_on="ADAUID", right_on="GEO_NAME", how="left")

# Keep only the columns we want: ADAUID, counts, percentages, geometry
final_columns = [
    "ADAUID",
    "arts_nocs_count",
    "arts_nocs_pct",
    "arts_naics_count",
    "arts_naics_pct",
    "geometry"
]
gdf_final = gdf_final[final_columns]

print(f"Final GeoDataFrame has {len(gdf_final)} rows and columns: {gdf_final.columns.tolist()}")

Final GeoDataFrame has 279 rows and columns: ['ADAUID', 'arts_nocs_count', 'arts_nocs_pct', 'arts_naics_count', 'arts_naics_pct', 'geometry']


In [15]:
# CELL 5 — Save the final GeoPackage

output_gpkg = "../data/ada/prelim-nocs-naics.gpkg"
gdf_final.to_file(output_gpkg, driver="GPKG")

print(f"Saved preliminary NOCS/NAICS ADA data to: {output_gpkg}")

Saved preliminary NOCS/NAICS ADA data to: ../data/ada/prelim-nocs-naics.gpkg
